# 📊 Notebook 01: EDA & Tiền xử lý
## Đề tài 5: Dự báo thời tiết — Szeged Hungary 2006-2016

**Mục tiêu**: Khám phá dữ liệu, xử lý 3 bẫy (Loud Cover/Precip Type/Pressure), thống kê trước-sau.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_config, resolve_path
from src.data.cleaner import WeatherCleaner

config = load_config()
print('✅ Config loaded')


## 1. Dữ liệu thô — Thống kê TRƯỚC tiền xử lý


In [ ]:
raw_path = resolve_path(config['data']['raw_path'])
df_raw = pd.read_csv(raw_path)
print(f'Shape thô: {df_raw.shape}')
print(f'\nMissing values TRƯỚC:')
print(df_raw.isnull().sum())
print(f'\nSố duplicate: {df_raw.duplicated().sum()}')
print(f'Pressure = 0: {(df_raw["Pressure (millibars)"] == 0).sum()}')
print(f'Loud Cover unique: {df_raw["Loud Cover"].unique()}')
df_raw.head()


## 2. Làm sạch dữ liệu (WeatherCleaner)


In [ ]:
cleaner = WeatherCleaner(config)
df_clean = cleaner.run()
cleaner.save(df_clean, fmt='parquet')


## 3. Thống kê SAU tiền xử lý


In [ ]:
print(f'Shape sau clean: {df_clean.shape}')
print(f'\nMissing values SAU:')
print(df_clean.isnull().sum())
print(f'\nDuplicate SAU: {df_clean.duplicated().sum()}')
print(f'\nPressure min SAU: {df_clean["Pressure (millibars)"].min():.2f}')
print(f'Pressure median SAU: {df_clean["Pressure (millibars)"].median():.2f}')


### Bảng so sánh Trước – Sau
| Mục | Trước | Sau |
|-----|-------|-----|
| Số dòng | 96,453 | 96,429 |
| Missing (Precip Type) | 517 | 0 |
| Pressure = 0 | 1,288 | 0 |
| Cột Loud Cover | Có (toàn 0) | Đã drop |
| Duplicate | 24 | 0 |


## 4. EDA — Biểu đồ phân bố


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
numeric_cols = ['Temperature (C)', 'Humidity', 'Wind Speed (km/h)',
                'Visibility (km)', 'Pressure (millibars)', 'Apparent Temperature (C)']
for ax, col in zip(axes.flatten(), numeric_cols):
    df_clean[col].hist(bins=50, ax=ax, alpha=0.7, edgecolor='white')
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
plt.suptitle('Phân bố các biến số liên tục', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Tương quan giữa các biến


In [ ]:
plt.figure(figsize=(10, 8))
corr_cols = ['Temperature (C)', 'Apparent Temperature (C)', 'Humidity',
             'Wind Speed (km/h)', 'Visibility (km)', 'Pressure (millibars)']
corr_matrix = df_clean[corr_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', linewidths=0.5, square=True)
plt.title('Ma trận Tương quan', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Phân bố Summary + Precip Type


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
df_clean['Summary'].value_counts().head(10).plot(kind='barh', ax=axes[0], color='#2196F3')
axes[0].set_title('Top 10 Loại Thời Tiết', fontsize=14, fontweight='bold')
df_clean['Precip Type'].value_counts().plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FF9800'])
axes[1].set_title('Loại Mưa', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
